# 02 · Baselines with no language model

Two methods that use no LLM, so everything later has something to beat.

- **Trained flat classifier:** TF-IDF and Omikuji, a tree-based extreme classifier from Kähler et al.'s comparison. It
  ignores the taxonomy. XR-Transformer, which leads that comparison, has no build for this Mac, so this is not their
  best trained model.
- **Retrieval only:** the k nearest labels by BGE-M3 embedding similarity, with label paths as the label text.

The number of labels per item is chosen on the development split, once per method and test set, and never on test data.
All methods read the first 500 words of a text.

**Pass marks, written before running.**
- The trained classifier beats retrieval only at the finest tier on both test sets. If not, something is broken.
- On HumSet, the trained classifier's F1 on subpillars should be in the region of the paper's published baselines
  (0.33 for FastText on 1D subpillars). This is a sense check on different splits of labels, not a replication.


In [1]:
import time

import pandas as pd

from tiered_taxonomy import experiment as ex
from tiered_taxonomy import runs
from tiered_taxonomy.config import data_root, load_config
from tiered_taxonomy.methods.trained import TrainedClassifier, choose_top_k
from tiered_taxonomy.metrics import top_k

root = data_root()
config = load_config()
words = config["text"]["input_words"]
seed = config["seed"]
samples = config["samples"]
results = {}

/Users/matthewharris/git/nvidia-autoontology-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def sample_for(data):
    if data.name == ex.EURLEX:
        settings = samples["eurlex"]
        return ex.sample_eurlex(
            data,
            settings["normal_documents"],
            settings["long_documents"],
            settings["long_threshold_words"],
            seed,
        )
    return ex.sample_humset(data, samples["humset"]["per_language"], seed)


def run_baselines(data):
    bench = runs.make_bench(data, root, config)
    sample = sample_for(data)
    dev = ex.pilot_subset(data.dev, 1000, seed)
    ids = [item.id for item in sample]

    # trained flat classifier, trained once: its rankings are saved and reused, like every finished step
    def train():
        started = time.monotonic()
        model = TrainedClassifier(config["trained"]["max_features"])
        model.fit([ex.input_text(item, words) for item in data.train], ex.gold_exact(data.train))
        trained_at = time.monotonic() - started
        dev_ranked = model.rank([ex.input_text(item, words) for item in dev], top_k=20)
        k, dev_f1 = choose_top_k(ex.gold_exact(dev), dev_ranked, config["trained"]["top_k_choices"])
        ranked = model.rank([ex.input_text(item, words) for item in sample], top_k=50)
        return k, dev_f1, trained_at, ranked

    baseline = runs.trained_baseline(root, data.name, ids, train)
    k, dev_f1, trained_at = baseline.k, baseline.dev_f1, baseline.train_seconds
    trained = [top_k(item, k) for item in baseline.ranked]
    ex.save_predictions(root, data.name, "trained", ids, trained)
    # retrieval only
    dev_vectors = bench.store.embed([ex.input_text(item, words) for item in dev])
    dev_hits = bench.index.search_many(dev_vectors, 20)
    retrieval_k, retrieval_dev = choose_top_k(
        ex.gold_exact(dev), dev_hits, config["trained"]["top_k_choices"]
    )
    retrieved = runs.retrieval_labels(bench, sample, retrieval_k)
    ex.save_predictions(root, data.name, "retrieval", ids, retrieved)
    return {
        "items": len(sample),
        "trained": {
            "k": k,
            "dev_f1": round(dev_f1, 4),
            "train_seconds": round(trained_at, 1),
            "scores": ex.evaluate(data, sample, trained),
        },
        "retrieval": {
            "k": retrieval_k,
            "dev_f1": round(retrieval_dev, 4),
            "scores": ex.evaluate(data, sample, retrieved),
        },
        "candidate_recall": runs.candidate_grid(
            bench, sample, bench.index, config["funnel"]["candidate_grid"]
        ),
    }


def table(result):
    rows = []
    for name in ("trained", "retrieval"):
        scores = result[name]["scores"]
        row = {"method": name, "k": result[name]["k"], "exact F1": round(scores["exact"]["f1"], 3)}
        row.update({f"tier {tier} F1": round(value["f1"], 3) for tier, value in scores["tiers"].items()})
        row.update({f"{band} recall": value["recall"] for band, value in scores["rarity"].items()})
        rows.append(row)
    return pd.DataFrame(rows)

## MultiEURLEX

In [3]:
results[ex.EURLEX] = run_baselines(ex.load_eurlex(root))
table(results[ex.EURLEX])

2026-11-11T22:38:17.885Z INFO  [omikuji::data] Loading data from /var/folders/9b/g2cxr52j3gz6bg6tpjjscd5w0000gn/T/tmpo16zchag/train.txt
2026-11-11T22:38:18.018Z INFO  [omikuji::data] Parsing data


2026-11-11T22:38:18.154Z INFO  [omikuji::data] Loaded 55000 examples; it took 0.27s
2026-11-11T22:38:18.228Z INFO  [omikuji::model::train] Training model with hyper-parameters HyperParam { n_trees: 3, min_branch_size: 100, max_depth: 20, centroid_threshold: 0.0, collapse_every_n_layers: 0, linear: HyperParam { loss_type: Hinge, eps: 0.1, c: 1.0, weight_threshold: 0.1, max_iter: 20 }, cluster: HyperParam { k: 2, balanced: true, eps: 0.0001, min_size: 2 }, tree_structure_only: false, train_trees_1_by_1: false }
2026-11-11T22:38:18.229Z INFO  [omikuji::model::train] Initializing tree trainer
2026-11-11T22:38:18.235Z INFO  [omikuji::model::train] Computing label centroids


2026-11-11T22:38:20.388Z INFO  [omikuji::model::train] Start training forest


67 / 12655 [>-----------------------------------------------] 0.53 % 55.46/s 4m 

137 / 12671 [>----------------------------------------------] 1.08 % 95.84/s 2m 

337 / 12689 [=>--------------------------------------------] 2.66 % 185.48/s 1m 

... (119 outputs left out)

12603 / 12999 [=========================================>-] 96.95 % 319.85/s 1s 

12999 / 12999 [=============================================] 100.00 % 326.90/s 

2026-11-11T22:39:00.153Z INFO  [omikuji::model::train] Model training complete; it took 41.92s


,method,k,exact F1,tier 1 F1,tier 2 F1,tier 3 F1,common recall,rare recall,unseen recall
0,trained,6,0.420,0.703,0.630,0.600,0.4369,0.0108,0.0
1,retrieval,10,0.112,0.499,0.337,0.255,0.1429,0.0968,0.2


In [4]:
pd.DataFrame(results[ex.EURLEX]["candidate_recall"]).T

,all,common,rare,unseen
5,0.0910,0.0904,0.0753,0.1333
10,0.1424,0.1429,0.0968,0.2000
20,0.1992,0.2019,0.1344,0.2167
50,0.3055,0.3112,0.2097,0.2833
100,0.4055,0.4119,0.3280,0.3333
200,0.5229,0.5273,0.4785,0.4583


## HumSet

In [5]:
results[ex.HUMSET] = run_baselines(ex.load_humset(root))
table(results[ex.HUMSET])

2026-11-11T22:39:27.007Z INFO  [omikuji::data] Loading data from /var/folders/9b/g2cxr52j3gz6bg6tpjjscd5w0000gn/T/tmpqv7wndmc/train.txt
2026-11-11T22:39:27.029Z INFO  [omikuji::data] Parsing data
2026-11-11T22:39:27.088Z INFO  [omikuji::data] Loaded 112037 examples; it took 0.08s
2026-11-11T22:39:27.109Z INFO  [omikuji::model::train] Training model with hyper-parameters HyperParam { n_trees: 3, min_branch_size: 100, max_depth: 20, centroid_threshold: 0.0, collapse_every_n_layers: 0, linear: HyperParam { loss_type: Hinge, eps: 0.1, c: 1.0, weight_threshold: 0.1, max_iter: 20 }, cluster: HyperParam { k: 2, balanced: true, eps: 0.0001, min_size: 2 }, tree_structure_only: false, train_trees_1_by_1: false }
2026-11-11T22:39:27.109Z INFO  [omikuji::model::train] Initializing tree trainer
2026-11-11T22:39:27.111Z INFO  [omikuji::model::train] Computing label centroids


2026-11-11T22:39:27.739Z INFO  [omikuji::model::train] Start training forest


186 / 186 [==================================================] 100.00 % 17.78/s 

2026-11-11T22:39:38.197Z INFO  [omikuji::model::train] Model training complete; it took 11.09s


,method,k,exact F1,tier 1 F1,tier 2 F1,common recall,rare recall
0,trained,3,0.538,0.586,0.456,0.6158,0.3710
1,retrieval,8,0.103,0.231,0.072,0.2221,0.5161


In [6]:
pd.DataFrame(results[ex.HUMSET]["candidate_recall"]).T

,all,common,rare
5,0.1649,0.1618,0.3387
10,0.2681,0.2631,0.5484
20,0.4497,0.4436,0.7903
50,0.8237,0.8217,0.9355
100,1.0000,1.0000,1.0000
200,1.0000,1.0000,1.0000


In [7]:
for name, result in results.items():
    finest = max(result["trained"]["scores"]["tiers"], key=int)
    if (
        result["trained"]["scores"]["tiers"][finest]["f1"]
        <= result["retrieval"]["scores"]["tiers"][finest]["f1"]
    ):
        raise ValueError(f"the trained classifier does not beat retrieval on {name}: check the baselines")
ex.save_results("02_baselines", results);